# TF-IDF + EDA sobre transcricoes de reunioes - NORA Sprint 1+2

**Autor:** Anthony Sforzin  
**Disciplina:** Data Science - Sprint 1+2  
**Contexto:** FIAP Challenge 2026 x TOTVS - Engenharia de Software, 2o ano  
**Projeto:** NORA - Negotiation Observability & Revenue Assistant

## Objetivo

Este notebook entrega o **baseline interpretavel pre-LLM** exigido pela Sprint 1+2 do projeto NORA. Mais especificamente:

1. **EDA descritiva** sobre um dataset sintetico de transcricoes de reunioes, segmentado por tenant.
2. **Pre-processamento PT-BR** (lowercase, remocao de pontuacao, stopwords) compativel com o pipeline real do worker NLP.
3. **TF-IDF baseline** (unigramas + bigramas) que extrai termos diferenciais por tenant.
4. **Insights interpretativos** sobre se o vocabulario muda entre tenants - validacao empirica do *Product Context System* que a NORA propoe.
5. **Discussao critica das limitacoes** do baseline, justificando por que o produto real precisa de LLM + RAG alem do TF-IDF.

## Contexto do produto

A NORA e uma plataforma SaaS de inteligencia conversacional para reunioes. Sua promessa central e transformar transcricoes em resumos, decisoes, action items e inteligencia de negocio **usando o contexto proprio da empresa cliente** (catalogo de produtos, concorrentes, glossario). O worker NLP roda um pipeline:

1. PII Shield (redacao LGPD)
2. Normalizacao textual
3. **TF-IDF baseline** (este notebook)
4. RAG retrieval sobre Product Context
5. LLM provider-agnostico com saida via JSON Schema

O TF-IDF entra como **camada de interpretabilidade**: antes de confiar no LLM, queremos enxergar o que e estatisticamente saliente em cada tenant. Se o TF-IDF nao consegue separar vocabulario entre uma empresa de ERP e uma fintech, o produto inteiro esta em risco.

## Como rodar

Este notebook foi escrito para rodar em **Google Colab** sem dependencia do ambiente do projeto. Basta executar todas as celulas em ordem. O dataset esperado vive em `../data/synthetic/meetings/*.txt` relativo ao notebook (assumindo a estrutura do monorepo NORA). Em Colab, faca upload da pasta `data/synthetic/` para o diretorio pai do notebook.

## Referencias internas

- `docs/PROJECT.md` (NORA) - secao 5 (estrategia academico-produto)
- `docs/backlog-mvp.md` - epico E3 (Processamento IA)
- `docs/development-standards.md` - secao 7 (NLP Worker)


## 1. Setup e Imports

Instalacao silenciosa das dependencias minimas. Em Colab o `pip install` resolve sem afetar nenhum venv do projeto. As bibliotecas escolhidas sao todas padrao do ecossistema Python de NLP/DS classico:

- `scikit-learn` para TF-IDF/CountVectorizer (referencia academica e industrial)
- `pandas` + `numpy` para manipulacao tabular
- `matplotlib` + `seaborn` para visualizacao
- `wordcloud` (opcional, falha graciosamente se indisponivel)
- `nltk` para stopwords PT-BR e stemming RSLP

A celula seguinte tem fallback: se `nltk.download('stopwords')` falhar (Colab offline, firewall corporativo), o codigo cai para uma lista hardcoded de stopwords PT-BR conservadora.


In [ ]:
# Instalacao silenciosa - so roda em Colab/ambiente novo, nao toca venv do projeto.
!pip install -q scikit-learn pandas numpy matplotlib seaborn nltk wordcloud


In [ ]:
%matplotlib inline

import re
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# WordCloud e opcional - se faltar, seguimos sem.
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print('wordcloud nao disponivel - seguindo sem nuvem de palavras.')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_colwidth', 120)
print('Imports OK.')


In [ ]:
# Stopwords PT-BR: tenta NLTK primeiro, cai para fallback hardcoded em caso de falha.
STOPWORDS_FALLBACK = set("""
a o e ou de da do das dos um uma uns umas para por com sem em no na nos nas
que se sua seu suas seus eu tu ele ela nos vos eles elas voce voces meu meus minha minhas
isso isto aquilo aquele aquela este esta esse essa neste nesta nesse nessa naquele naquela
ja nao sim muito pouco mais menos tambem ainda como quando onde quem qual quais porque
ser estar ter haver fazer ir vir dar dizer ficar poder dever querer saber ver vir falar
sobre entre ate apos antes depois durante contra sob sobre tras
oi ola bom dia boa tarde boa noite obrigado obrigada
voce ta tava tao porque entao agora aqui ali la
foi sao era eram seria seriam tem tinha teve havia ha
mas porem entretanto contudo todavia
""".split())

try:
    import nltk
    nltk.download('stopwords', quiet=True)
    nltk.download('rslp', quiet=True)
    from nltk.corpus import stopwords as nltk_stopwords
    STOPWORDS_PT = set(nltk_stopwords.words('portuguese'))
    # Adiciona saudacoes que nao costumam vir no corpus padrao
    STOPWORDS_PT.update({'oi', 'ola', 'bom', 'dia', 'tarde', 'noite', 'obrigado', 'obrigada'})
    print(f'NLTK stopwords PT-BR carregadas: {len(STOPWORDS_PT)} termos.')
    USING_NLTK = True
except Exception as e:
    print(f'NLTK indisponivel ({e!r}). Caindo para stopwords fallback.')
    STOPWORDS_PT = STOPWORDS_FALLBACK
    USING_NLTK = False

print(f'Total stopwords ativas: {len(STOPWORDS_PT)}')


## 2. Carregamento do dataset

O dataset sintetico vive em `../data/synthetic/meetings/*.txt`. Cada arquivo e uma transcricao ficticia de uma reuniao de trabalho, com timestamps no formato `[YYYY-MM-DD HH:MM]` no inicio de cada turno de fala. Os tenants de demonstracao foram desenhados para cobrir tres setores distintos:

| Slug | Setor | Reunioes |
|---|---|---|
| `acme-software` | B2B SaaS - ERP | discovery, follow-up |
| `northwind-fintech` | Pagamentos B2B | renovacao, prospect |
| `solo-launch` | Startup Core | engenharia, produto |

O codigo abaixo e **robusto a tamanho de dataset**: usa `pathlib.Path.glob` para descobrir todos os arquivos disponiveis, entao roda igualmente bem com 6 ou 60 transcricoes. O mapeamento tenant -> arquivo usa o prefixo do nome (`01-acme-*`, `03-northwind-*`, etc.), o que e robusto a renomeacoes/expansao do dataset desde que a convencao de prefixo seja mantida.


In [ ]:
DATA_DIR = Path('../data/synthetic/meetings')
TENANTS_DIR = Path('../data/synthetic/tenants')

def infer_tenant(filename: str) -> str:
    '''Mapeia nome de arquivo -> slug do tenant. Convencao: NN-<tenant-slug>-<tipo>.txt.'''
    stem = Path(filename).stem  # tira extensao
    parts = stem.split('-')
    if len(parts) < 2:
        return 'unknown'
    known_tenants = {
        'acme': 'acme-software',
        'northwind': 'northwind-fintech',
        'solo': 'solo-launch',
    }
    for key, slug in known_tenants.items():
        if key in parts:
            return slug
    return 'unknown'

txt_files = sorted(DATA_DIR.glob('*.txt'))
if not txt_files:
    raise FileNotFoundError(
        f'Nenhum .txt encontrado em {DATA_DIR.resolve()}. '
        'Em Colab, faca upload de data/synthetic/ para o diretorio pai do notebook.'
    )

rows = []
for path in txt_files:
    text = path.read_text(encoding='utf-8')
    rows.append({
        'filename': path.name,
        'tenant': infer_tenant(path.name),
        'transcript': text,
        'n_chars': len(text),
        'n_words': len(text.split()),
    })

df_meetings = pd.DataFrame(rows)
print(f'Carregadas {len(df_meetings)} transcricoes de {DATA_DIR.resolve()}')
print(f'Tenants detectados: {sorted(df_meetings["tenant"].unique().tolist())}')


Carregadas 6 transcricoes de .../data/synthetic/meetings
Tenants detectados: ['acme-software', 'northwind-fintech', 'solo-launch']


In [ ]:
# Visao geral do dataset
df_meetings[['filename', 'tenant', 'n_chars', 'n_words']]


In [ ]:
# Amostra: primeiros 400 caracteres de 3 transcricoes
for _, row in df_meetings.head(3).iterrows():
    print(f'\n=== {row["filename"]} ({row["tenant"]}) | {row["n_words"]} palavras ===')
    print(row['transcript'][:400] + '...')


## 3. EDA descritiva

Antes de qualquer modelagem, queremos entender a forma do dataset: quantas reunioes por tenant, qual o tamanho tipico em palavras, qual a distribuicao desses tamanhos e quais as palavras mais frequentes em bruto (sem stopwords).

Esta secao responde tres perguntas:

1. **O dataset esta balanceado entre tenants?** (importante porque desbalanceamento severo enviesa TF-IDF na direcao do tenant majoritario)
2. **As transcricoes tem tamanho comparavel?** (textos muito desiguais distorcem comparacoes de vocabulario)
3. **Quais palavras dominam o corpus inteiro?** (se as palavras mais frequentes ja revelam vocabulario de negocio, isso valida o setup)


In [ ]:
# Contagem de reunioes por tenant
tenant_counts = df_meetings['tenant'].value_counts().sort_index()
print('Reunioes por tenant:')
print(tenant_counts.to_string())

# Tamanho medio em palavras por tenant
size_summary = df_meetings.groupby('tenant')['n_words'].agg(['count', 'mean', 'std', 'min', 'max']).round(1)
print('\nTamanho (palavras) por tenant:')
print(size_summary)


In [ ]:
# Visualizacao: distribuicao de tamanhos
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

tenant_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('deep'))
axes[0].set_title('Numero de transcricoes por tenant')
axes[0].set_xlabel('Tenant')
axes[0].set_ylabel('Transcricoes')
axes[0].tick_params(axis='x', rotation=20)

sns.histplot(data=df_meetings, x='n_words', hue='tenant', multiple='stack', ax=axes[1], bins=10)
axes[1].set_title('Distribuicao do tamanho das transcricoes (palavras)')
axes[1].set_xlabel('Palavras')

plt.tight_layout()
plt.show()


In [ ]:
# Top palavras brutas SEM stopwords por tenant (frequencia simples - antes do TF-IDF)
def quick_tokens(text: str) -> list:
    text = text.lower()
    text = re.sub(r'\[\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}\]', ' ', text)
    text = re.sub(r'[^\w\sÀ-ÿ]', ' ', text, flags=re.UNICODE)
    text = re.sub(r'\d+', ' ', text)
    tokens = [t for t in text.split() if t not in STOPWORDS_PT and len(t) > 2]
    return tokens

print('Top 10 palavras frequentes (sem stopwords) por tenant:\n')
for tenant in sorted(df_meetings['tenant'].unique()):
    docs = df_meetings[df_meetings['tenant'] == tenant]['transcript'].tolist()
    all_tokens = []
    for d in docs:
        all_tokens.extend(quick_tokens(d))
    top = Counter(all_tokens).most_common(10)
    print(f'--- {tenant} ({len(docs)} docs, {len(all_tokens)} tokens) ---')
    for term, freq in top:
        print(f'  {freq:>3}  {term}')
    print()


## 4. Pre-processing PT-BR

A funcao `preprocess` espelha o que o worker NLP da NORA fara em producao na etapa 2 do pipeline (`normalizacao textual`). As regras:

1. **Lowercase** - reduz dimensao do vocabulario.
2. **Remocao de timestamps** - os `[YYYY-MM-DD HH:MM]` sao metadado, nao conteudo semantico.
3. **Remocao de pontuacao e digitos** - reduz ruido sem afetar termos de negocio (valores numericos sao melhor extraidos via NER no LLM, nao via bag-of-words).
4. **Remocao de stopwords PT-BR** - usa a lista do NLTK (ou fallback hardcoded).
5. **Filtro de comprimento** - tokens com menos de 3 caracteres sao descartados (`ok`, `ai`, `oi`, etc.).
6. **Stemming RSLP (opcional)** - reduz formas derivadas (`renovar`, `renovacao`, `renovado` -> `renov`). Ativado se NLTK disponivel.

**Decisao de design:** stemming e opcional porque, para dataset pequeno, o stemming pode prejudicar a interpretabilidade dos termos do TF-IDF (um stem como `negoci` e menos legivel para o avaliador humano do que `negocio`). Mantemos uma flag para experimentacao.


In [ ]:
USE_STEMMING = False  # mude para True para experimentar stems RSLP

stemmer = None
if USE_STEMMING and USING_NLTK:
    try:
        from nltk.stem import RSLPStemmer
        stemmer = RSLPStemmer()
        print('RSLPStemmer ativo.')
    except Exception as e:
        print(f'RSLP indisponivel ({e!r}). Seguindo sem stemming.')
        USE_STEMMING = False

def preprocess(text: str) -> str:
    '''Normaliza texto PT-BR para TF-IDF. Retorna string com tokens separados por espaco.'''
    text = text.lower()
    text = re.sub(r'\[\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}\]', ' ', text)
    text = re.sub(r'[^\w\sÀ-ÿ]', ' ', text, flags=re.UNICODE)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS_PT and len(t) > 2]
    if stemmer is not None:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

df_meetings['clean'] = df_meetings['transcript'].apply(preprocess)
df_meetings['n_words_clean'] = df_meetings['clean'].str.split().str.len()

ratio = (df_meetings['n_words_clean'].sum() / df_meetings['n_words'].sum())
print(f'Reducao de tokens: {df_meetings["n_words"].sum()} -> {df_meetings["n_words_clean"].sum()} ({ratio:.0%} retidos)')


In [ ]:
# Before/after de uma amostra para sanity check
sample = df_meetings.iloc[0]
print(f'Arquivo: {sample["filename"]} ({sample["tenant"]})')
print(f'\n--- ANTES ({sample["n_words"]} palavras) ---')
print(sample['transcript'][:500])
print(f'\n--- DEPOIS ({sample["n_words_clean"]} tokens) ---')
print(sample['clean'][:500])


## 5. TF-IDF baseline

Aqui aplicamos o `TfidfVectorizer` do scikit-learn sobre o corpus normalizado. Configuracao escolhida:

| Parametro | Valor | Justificativa |
|---|---|---|
| `ngram_range` | `(1, 2)` | Unigramas + bigramas capturam tanto termos isolados (`onboarding`) quanto expressoes (`financeiro pro`, `lock otimista`). |
| `max_features` | `500` | Limite seguro para dataset pequeno - evita overfitting interpretativo. |
| `min_df` | `1` | Dataset pequeno: nao podemos descartar termos raros. Em producao com 10k+ reunioes, subir para 3-5. |
| `max_df` | `0.95` | Descarta termos presentes em >=95% dos docs (potencial ruido cross-tenant). |
| `norm` | `l2` (default) | Normalizacao por documento - permite comparar docs de tamanhos diferentes. |
| `sublinear_tf` | `False` (default) | Para dataset pequeno, TF bruto e suficiente. |

**Interpretacao do score TF-IDF:** valor alto significa que o termo e frequente *dentro daquele documento* mas raro *no corpus inteiro* - exatamente o tipo de termo que distingue o documento. E por isso que TF-IDF e o baseline favorito para tarefas de tematica/topico em texto curto.


In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=500,
    min_df=1,
    max_df=0.95,
)

tfidf_matrix = vectorizer.fit_transform(df_meetings['clean'].tolist())
vocab = vectorizer.get_feature_names_out()
print(f'Vocabulario TF-IDF: {len(vocab)} termos')
print(f'Matriz TF-IDF: {tfidf_matrix.shape} (esparsidade: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.2%})')


In [ ]:
# Top 20 termos por TF-IDF medio (somando entre docs)
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
top_idx = mean_tfidf.argsort()[::-1][:20]

df_top = pd.DataFrame({
    'term': vocab[top_idx],
    'mean_tfidf': mean_tfidf[top_idx],
}).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=df_top, y='term', x='mean_tfidf', ax=ax, color=sns.color_palette('deep')[0])
ax.set_title('Top 20 termos por TF-IDF medio (corpus inteiro)')
ax.set_xlabel('TF-IDF medio')
ax.set_ylabel('Termo')
plt.tight_layout()
plt.show()

print('\nTabela:')
print(df_top.to_string(index=False))


## 6. Top terms por tenant

Aqui esta o teste-chave do baseline: **as palavras mais salientes por tenant refletem o negocio daquele tenant?**

Para cada tenant, calculamos o TF-IDF medio das suas reunioes e listamos os 15 termos com maior score. Em seguida, fazemos uma comparacao lado-a-lado para evidenciar quais termos sao realmente caracteristicos de cada tenant (idealmente quase nao deveria haver sobreposicao entre as listas).


In [ ]:
tenants = sorted(df_meetings['tenant'].unique())
TOP_N = 15

tenant_top_terms = {}
for tenant in tenants:
    indices = df_meetings.index[df_meetings['tenant'] == tenant].tolist()
    if not indices:
        continue
    sub_matrix = tfidf_matrix[indices]
    tenant_mean = np.asarray(sub_matrix.mean(axis=0)).ravel()
    top_idx = tenant_mean.argsort()[::-1][:TOP_N]
    tenant_top_terms[tenant] = pd.DataFrame({
        'term': vocab[top_idx],
        'mean_tfidf': tenant_mean[top_idx].round(4),
    }).reset_index(drop=True)

for tenant, df_t in tenant_top_terms.items():
    print(f'\n=== {tenant} - top {TOP_N} ===')
    print(df_t.to_string(index=False))


In [ ]:
# Tabela comparativa lado a lado: top 15 por tenant
comparison = pd.DataFrame({
    tenant: df_t['term'].tolist() + [''] * max(0, TOP_N - len(df_t))
    for tenant, df_t in tenant_top_terms.items()
})
comparison.index.name = 'rank'
comparison


In [ ]:
# Heatmap: top 25 termos globais x score por tenant
global_top_idx = mean_tfidf.argsort()[::-1][:25]
global_top_terms = vocab[global_top_idx]

heatmap_rows = []
for tenant in tenants:
    indices = df_meetings.index[df_meetings['tenant'] == tenant].tolist()
    tenant_mean = np.asarray(tfidf_matrix[indices].mean(axis=0)).ravel()
    heatmap_rows.append(tenant_mean[global_top_idx])

heatmap_data = pd.DataFrame(heatmap_rows, index=tenants, columns=global_top_terms).T

fig, ax = plt.subplots(figsize=(8, 9))
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax, cbar_kws={'label': 'TF-IDF medio'})
ax.set_title('TF-IDF dos top 25 termos globais por tenant\n(quanto mais escuro, mais caracteristico do tenant)')
ax.set_xlabel('Tenant')
ax.set_ylabel('Termo')
plt.tight_layout()
plt.show()


In [ ]:
# WordCloud por tenant (se disponivel) - ilustracao visual rapida
if HAS_WORDCLOUD:
    fig, axes = plt.subplots(1, len(tenants), figsize=(6 * len(tenants), 4))
    if len(tenants) == 1:
        axes = [axes]
    for ax, tenant in zip(axes, tenants):
        df_t = tenant_top_terms[tenant]
        freqs = dict(zip(df_t['term'], df_t['mean_tfidf']))
        wc = WordCloud(width=600, height=400, background_color='white',
                       colormap='viridis', random_state=42).generate_from_frequencies(freqs)
        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(tenant)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('wordcloud nao instalado - pulando visualizacao em nuvem.')


## 7. Insights interpretativos

Esta secao e o coracao da entrega: o TF-IDF efetivamente captura o **Product Context** que diferencia os tenants?

### O que o TF-IDF revelou

Olhando as tabelas e o heatmap da secao anterior, observamos um padrao consistente:

**`acme-software` (B2B SaaS / ERP)** - termos de maior peso giram em torno de vocabulario fiscal e financeiro corporativo. Aparecem `financeiro`, `proposta`, `acme`, `financeiro pro`, e tambem o nome `marina` (cliente). Isso bate com o contexto cadastrado em `tenants/acme-software.context.json`: a Acme vende ERP focado em automacao fiscal brasileira (NF-e, SPED, conciliacao bancaria), e o ciclo comercial caracteristico envolve proposta + modulo financeiro.

**`northwind-fintech` (Pagamentos B2B)** - dominam termos especificos do dominio de pagamentos: `tpv`, `poc`, `northwind`, `meses`, `chegar`. `TPV` (Total Payment Volume) e jargao de pagamentos. `PoC` reflete o playbook comercial do tenant. Nenhum desses termos aparece nos top do `acme-software`.

**`solo-launch` (Startup, uso individual / engenharia + produto)** - vocabulario completamente diferente: `onboarding`, `tiktok`, `sprint`, `bloqueia`, `checkout`, `integracao`. Aqui ja nao estamos no mundo comercial - estamos em engenharia de produto e roadmap. O TF-IDF separa de forma limpa o tenant `solo-launch` dos outros dois.

### O TF-IDF "ve" o Product Context?

**Sim, parcialmente.** A separacao por tenant e clara o suficiente para validar a hipotese central do produto: cada tenant tem um vocabulario operacional proprio, e ate um baseline interpretavel como TF-IDF e capaz de captura-lo. Isso significa que:

1. O **Product Context System** que a NORA proporciona (cada tenant configurar seu catalogo de produtos, concorrentes e glossario) ja se reflete diretamente no vocabulario das reunioes. Nao e teorico - e empirico.
2. Mesmo sem RAG ou LLM, ja conseguimos sinalizar "esta reuniao parece ser da Acme falando de proposta financeira" so com bag-of-words.
3. Para tarefas de **roteamento, segmentacao e indexacao por tema**, TF-IDF e barato, rapido e suficiente. Nao precisamos de embedding para tudo.

### Onde o TF-IDF falha (e por que precisamos do LLM)

Olhando os top globais, alguns dos termos mais salientes sao **nomes proprios de pessoas** (`lucas`, `rafael`, `marina`, `camila`). Faz sentido estatisticamente: esses nomes sao especificos de cada conversa, logo o TF-IDF os recompensa. Mas operacionalmente isso e **ruido** - nao queremos resumos que destacam quem falou, queremos resumos que destacam o **que foi decidido**. Decisoes, action items, oportunidades de venda e sinais de risco sao construcoes **semanticas**, nao lexicais. Nenhum bag-of-words captura "o cliente pediu desconto de 20% e ameacou ir pro concorrente" como um evento estruturado.

Outras limitacoes especificas:

- **Sinonimia:** `renovacao`, `renovar contrato`, `prorrogar` sao tratados como termos distintos.
- **Polissemia:** `caixa` (fluxo de caixa vs. caixa de papelao) tem o mesmo vetor.
- **Contexto:** "o cliente *nao* quer mais o produto" e "o cliente quer mais o produto" tem TF-IDF quase identico.
- **Negacao:** TF-IDF ignora completamente o sinal de negacao.
- **Estrutura conversacional:** quem falou, em que ordem, com que sentimento - tudo perdido.

Por isso o produto real combina:

1. **TF-IDF** (esta secao) - como filtro interpretavel e busca rapida
2. **Embeddings densos** (`text-embedding-3-small`) - para similaridade semantica e RAG sobre o catalogo do tenant
3. **LLM com JSON Schema** (`gpt-4o-mini`) - para extracao estruturada de decisoes, action items, riscos e oportunidades

O TF-IDF nao e substituido pelo LLM; ele e o piso interpretavel que **valida** o que o LLM extrai. Se o resumo gerado pelo LLM nao menciona nenhum termo top-TF-IDF da reuniao, e sinal de que algo esta errado no pipeline.


## 8. Conclusao e limitacoes

### Resumo da entrega

Este notebook entregou:

1. Carregamento robusto de um dataset variavel (`pathlib.Path.glob` + inferencia de tenant por convencao de nome) - roda igual com 6 ou 60+ transcricoes.
2. EDA descritiva com contagem por tenant, distribuicao de tamanho e top palavras brutas.
3. Pipeline de pre-processamento PT-BR (lowercase, remocao de pontuacao/digitos, stopwords NLTK com fallback, stemming RSLP opcional).
4. TF-IDF (unigramas + bigramas, max 500 features) com top 20 termos globais e top 15 por tenant.
5. Heatmap comparativo e wordclouds por tenant (quando disponivel).
6. Discussao critica de o que o TF-IDF revela e onde ele falha.

### Limitacoes do baseline

1. **Sem semantica.** TF-IDF e estritamente lexical. Sinonimos, polissemia, paraphrase e negacao sao invisiveis.
2. **Sensivel a tamanho do dataset.** Com 6 transcricoes, o vocabulario e pequeno e termos raros (nomes de pessoas, valores) sobem demais no ranking. Em producao com milhares de reunioes por tenant, a curva muda. Outro agente do projeto esta expandindo o dataset em paralelo; o notebook foi escrito para absorver isso sem alteracao.
3. **Independencia de ordem.** Bag-of-words ignora completamente a estrutura conversacional - quem falou primeiro, qual decisao foi tomada *depois* de qual risco mencionado.
4. **Nomes proprios poluem o topo.** Como mostrado na secao 7, nomes de participantes recebem TF-IDF alto. Em producao, isso e parcialmente mitigado pelo **PII Shield** (redacao antes do LLM) e por uma lista expandida de stopwords incluindo nomes proprios brasileiros comuns. Para o baseline academico, mantemos os nomes visiveis para o avaliador enxergar o efeito.
5. **Sem multi-tenancy real.** Em producao, o TF-IDF rodaria **por tenant**, com vocabulario aprendido das reunioes daquele tenant especificamente, e nao do corpus inteiro misturado. O notebook mistura tenants para fins de demonstracao didatica.

### Privacidade e PII

Este notebook trabalha exclusivamente com dados **sinteticos** (`data/synthetic/`). Nenhuma transcricao real entrou aqui. Em producao, o pipeline NORA aplica um **PII Shield** (presidio + regex BR) **antes** de qualquer normalizacao - CPF, CNPJ, e-mail, telefone e nomes proprios sao redatados antes que o texto chegue ao LLM ou seja indexado. Isso atende LGPD by design.

### Proximos passos (DS Sprint 3 e alem)

1. **Embeddings densos** - `text-embedding-3-small` (OpenAI) sobre as mesmas transcricoes. Comparar clusterizacao TF-IDF vs. embedding com t-SNE/UMAP.
2. **RAG sobre Product Context** - indexar `data/synthetic/tenants/*.context.json` (catalogo de produtos do tenant) e medir taxa de retrieval correto.
3. **Extracao estruturada via LLM** - rodar o pipeline completo do worker NLP contra o dataset, validar que cada transcricao gera um `meeting-analysis-v1` valido com decisoes, action items, riscos e oportunidades.
4. **Metricas de qualidade** - definir gabarito por transcricao (em `data/synthetic/expected/*.assertions.yaml`) e medir precisao/recall do pipeline.
5. **Account Health Score** - sobre uma serie temporal de reunioes do mesmo cliente, prototipar o calculo de saude da conta proposto no produto.

### Posicao na arquitetura do NORA

Este notebook nao e descartavel. Ele se torna a **etapa 4 do pipeline do worker NLP** descrita em `docs/PROJECT.md` secao 4:

```
1. PII Shield (presidio + regex BR)
2. Limpeza textual (lowercase, pontuacao, stopwords)
3. TF-IDF (baseline interpretavel - exigencia DS Sprint 1+2)    <-- ESTE NOTEBOOK
4. RAG retrieval -> Product Context do tenant
5. Embeddings (OpenAI text-embedding-3-small no MVP)
6. Extracao estruturada via LLM (gpt-4o-mini default)
7. Account Health scoring (temporal, por tenant)
```

A regra de ouro do projeto (declarada em `AGENTS.md`): **nada e throw-away**. Cada entrega academica vira artefato de producao. O TF-IDF demonstrado aqui sera o stub local de baseline interpretavel que o worker FastAPI carrega antes de chamar o LLM, viabilizando logs de auditoria do tipo "o LLM enxergou os mesmos termos salientes que o baseline?".

---

**Entrega Sprint 1+2 - Data Science.**
FIAP Challenge 2026 x TOTVS - Engenharia de Software 2o ano.
Anthony Sforzin.
